In [1]:
# ============================================================
# 1- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [2]:
# ============================================================
# 2- Import Excel Data into the DataFrame
# ============================================================

from pathlib import Path
import pandas as pd

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)

# ------------------------------------------------------------
# 2.1 - Validate the Excel file
# ------------------------------------------------------------

if not EXCEL_FILE.is_file():
    raise FileNotFoundError(
        f"Excel file not found: {EXCEL_FILE}"
    )

# ------------------------------------------------------------
# 2.2 - Read Excel data
# ------------------------------------------------------------

df = pd.read_excel(
    EXCEL_FILE,
    header=None,
    usecols=[0, 1],
    dtype=str,
    keep_default_na=False,
    engine="openpyxl",
)

df.columns = ["full_name", "phone"]

# ------------------------------------------------------------
# 2.3 - Clean the imported data
# ------------------------------------------------------------

df["full_name"] = df["full_name"].str.strip()
df["phone"] = df["phone"].str.strip()

# Remove completely empty rows.
df = df.loc[
    ~(df["full_name"].eq("") & df["phone"].eq(""))
].copy()

df.reset_index(drop=True, inplace=True)

# ------------------------------------------------------------
# 2.4 - Normalize and validate phone numbers
# ------------------------------------------------------------

normalized_phones = []
validation_errors = []

for phone in df["phone"]:
    try:
        normalized_phone = normalize_iran_phone(phone)
        normalized_phones.append(normalized_phone)
        validation_errors.append(None)

    except ValueError as exc:
        normalized_phones.append(None)
        validation_errors.append(str(exc))

df["phone"] = normalized_phones
df["validation_error"] = validation_errors

# ------------------------------------------------------------
# 2.5 - Separate valid and invalid contact records
# ------------------------------------------------------------

invalid_contacts = df.loc[
    df["validation_error"].notna()
].copy()

valid_contacts = df.loc[
    df["validation_error"].isna()
].copy()

# ------------------------------------------------------------
# 2.6 - Identify duplicate phone numbers
# ------------------------------------------------------------

valid_contacts["is_duplicate"] = valid_contacts[
    "phone"
].duplicated(keep="first")

# ------------------------------------------------------------
# 2.7 - Assign contact IDs
# ------------------------------------------------------------

valid_contacts = valid_contacts.reset_index(drop=True)

valid_contacts.insert(
    0,
    "contact_id",
    range(len(valid_contacts))
)

# ------------------------------------------------------------
# 2.8 - Generate the final DataFrame
# ------------------------------------------------------------

contacts_df = valid_contacts[
    ["contact_id", "full_name", "phone"]
].copy()

# Make df the final DataFrame with exactly three columns.
df = contacts_df.copy()

# ------------------------------------------------------------
# 2.9 - Display the import summary
# ------------------------------------------------------------

print("=" * 60)
print("EXCEL CONTACT IMPORT SUMMARY")
print("=" * 60)

print(f"Excel file:             {EXCEL_FILE.name}")
print(f"Imported records:       {len(invalid_contacts) + len(valid_contacts)}")
print(f"Valid contacts:         {len(contacts_df)}")
print(f"Invalid records:        {len(invalid_contacts)}")
print(
    "Duplicate phone rows:   "
    f"{int(valid_contacts['is_duplicate'].sum())}"
)

# ------------------------------------------------------------
# 2.10 - Display invalid records for review
# ------------------------------------------------------------

if not invalid_contacts.empty:
    print("\\nInvalid records requiring review:")

    display(
        invalid_contacts[
            ["full_name", "phone", "validation_error"]
        ]
    )

# ------------------------------------------------------------
# 2.11 - Optional: display duplicate contacts
# ------------------------------------------------------------

duplicates_df = valid_contacts.loc[
    valid_contacts["is_duplicate"]
].copy()

if not duplicates_df.empty:
    print("\\nDuplicate phone numbers requiring review:")
    display(duplicates_df)

EXCEL CONTACT IMPORT SUMMARY
Excel file:             contacts-1-44.xlsx
Imported records:       44
Valid contacts:         44
Invalid records:        0
Duplicate phone rows:   0


In [ ]:
df

In [3]:
# ============================================================
# 3- Selenium Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


In [ ]:
# ============================================================
# 4- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

In [4]:
# ============================================================
# 5- Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By


class Selectors:

    # 1. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.XPATH,
        "//button[contains(@class, 'btn-circle') "
        "and contains(@class, 'btn-corner') "
        "and .//span[@data-icon='add']]"
    )

    # 2. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 3. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 4. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 5. Toast shown when the phone number is not registered on Eitaa
    UNREGISTERED_TOAST = (
        By.CSS_SELECTOR,
        "div.toast.is-visible"
    )

    # 6. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [ ]:
DELETE
ME!
# ============================================================
# TEST - Section 5: Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

WAIT_TIMEOUT = 10

# Test number that produces the Eitaa "not registered" toast
TEST_UNREGISTERED_PHONE = "9111111111"

wait = WebDriverWait(driver, WAIT_TIMEOUT)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def verify_visible(name, locator):
    """
    Verify that a selector finds a visible element.
    """
    try:
        element = wait.until(
            EC.visibility_of_element_located(locator)
        )

        print(f"✓ {name}")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED\n"
            f"  Locator: {locator}"
        )


def click_element(name, locator):
    """
    Wait for an element to be clickable and click it.
    """
    try:
        element = wait.until(
            EC.element_to_be_clickable(locator)
        )

        element.click()
        print(f"✓ {name} - clicked")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED - element was not clickable\n"
            f"  Locator: {locator}"
        )


# ============================================================
# 1. Verify HAMBURGER_MENU
# ============================================================

print("=" * 70)
print("SECTION 6 - SELENIUM SELECTOR TEST")
print("=" * 70)

print("\n[1] HAMBURGER_MENU")
print("-" * 70)

verify_visible(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)

# ============================================================
# 2. Open hamburger menu
# ============================================================

print("\n[2] Open Hamburger Menu")
print("-" * 70)

click_element(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)

# ============================================================
# 3. Verify CONTACTS_BUTTON
# ============================================================

print("\n[3] CONTACTS_BUTTON")
print("-" * 70)

verify_visible(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)

# ============================================================
# 4. Open Contacts
# ============================================================

print("\n[4] Open Contacts")
print("-" * 70)

click_element(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)

# ============================================================
# 5. Verify ADD_CONTACT_BUTTON
# ============================================================

print("\n[5] ADD_CONTACT_BUTTON")
print("-" * 70)

verify_visible(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)

# ============================================================
# 6. Open Add Contact dialog
# ============================================================

print("\n[6] Open Add Contact Dialog")
print("-" * 70)

click_element(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)

# ============================================================
# 7. Verify FIRST_NAME
# ============================================================

print("\n[7] FIRST_NAME")
print("-" * 70)

first_name = verify_visible(
    "FIRST_NAME",
    Selectors.FIRST_NAME
)

# ============================================================
# 8. Verify PHONE
# ============================================================

print("\n[8] PHONE")
print("-" * 70)

phone = verify_visible(
    "PHONE",
    Selectors.PHONE
)

# ============================================================
# 9. Verify SAVE_BUTTON
# ============================================================

print("\n[9] SAVE_BUTTON")
print("-" * 70)

verify_visible(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)

# ============================================================
# 10. Verify CLOSE_CONTACT_WINDOW
# ============================================================

print("\n[10] CLOSE_CONTACT_WINDOW")
print("-" * 70)

verify_visible(
    "CLOSE_CONTACT_WINDOW",
    Selectors.CLOSE_CONTACT_WINDOW
)

# ============================================================
# 11. Enter test phone number
# ============================================================

print("\n[11] Enter Test Phone Number")
print("-" * 70)

phone.click()
phone.send_keys(TEST_UNREGISTERED_PHONE)

print(
    f"✓ PHONE accepted test number: "
    f"{TEST_UNREGISTERED_PHONE}"
)

# ============================================================
# 12. Enter test first name
# ============================================================

print("\n[12] Enter Test First Name")
print("-" * 70)

first_name.click()
first_name.send_keys("SelectorTest")

print("✓ FIRST_NAME accepted test text")

# ============================================================
# 13. Click SAVE
# ============================================================

print("\n[13] SAVE_BUTTON")
print("-" * 70)

click_element(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)

# ============================================================
# 14. Detect transient UNREGISTERED_TOAST
#
# IMPORTANT:
# The toast exists only briefly, so we poll the DOM
# frequently instead of using a long normal WebDriverWait.
# ============================================================

print("\n[14] UNREGISTERED_TOAST")
print("-" * 70)

toast_found = False

toast_locator = Selectors.UNREGISTERED_TOAST

toast_deadline = time.monotonic() + 5

while time.monotonic() < toast_deadline:

    try:
        toast_elements = driver.find_elements(
            *toast_locator
        )

        for toast in toast_elements:

            if toast.is_displayed():

                toast_found = True

                print("✓ UNREGISTERED_TOAST appeared")

                try:
                    print(
                        f"  Toast text: "
                        f"{toast.text!r}"
                    )
                except Exception:
                    pass

                break

        if toast_found:
            break

    except Exception:
        pass

    time.sleep(0.05)

if not toast_found:
    raise AssertionError(
        "✗ UNREGISTERED_TOAST FAILED\n"
        "The toast did not appear within the 5-second "
        "monitoring window."
    )

# ============================================================
# 15. CLOSE TOAST
#
# The same CLOSE_CONTACT_WINDOW selector is used.
# The first click dismisses the toast.
# ============================================================

print("\n[15] Close Toast")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - first click (dismiss toast)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Toast dismissed")

# ------------------------------------------------------------
# Give Eitaa a very short moment to update the UI.
# ------------------------------------------------------------

time.sleep(0.2)

# ============================================================
# 16. CLOSE ADD CONTACT WINDOW
#
# The same selector must be clicked a second time.
# ============================================================

print("\n[16] Close Add Contact Window")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - second click (close window)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Add Contact window close action executed")

# ============================================================
# 17. Verify Add Contact window is actually closed
# ============================================================

print("\n[17] Verify Add Contact Window Closed")
print("-" * 70)

try:
    WebDriverWait(driver, 5).until(
        EC.invisibility_of_element_located(
            Selectors.CLOSE_CONTACT_WINDOW
        )
    )

    print("✓ Add Contact window is closed")

except TimeoutException:
    raise AssertionError(
        "✗ Add Contact window is still visible after "
        "the second CLOSE_CONTACT_WINDOW click."
    )

# ============================================================
# 18. Final result
# ============================================================

print("\n" + "=" * 70)
print("SECTION 6 TEST PASSED")
print("=" * 70)

print("""
All Section 6 Selenium selectors were successfully verified:

✓ HAMBURGER_MENU
✓ CONTACTS_BUTTON
✓ ADD_CONTACT_BUTTON
✓ FIRST_NAME
✓ PHONE
✓ SAVE_BUTTON
✓ UNREGISTERED_TOAST
✓ CLOSE_CONTACT_WINDOW

The transient UNREGISTERED_TOAST was detected successfully.

Cleanup completed:

✓ First CLOSE_CONTACT_WINDOW click dismissed the toast
✓ Second CLOSE_CONTACT_WINDOW click closed the Add Contact window
✓ Add Contact window disappearance was verified
""")


In [5]:
# ============================================================
# 6- Reliable waits
# ============================================================

# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element exists in the DOM
# ------------------------------------------------------------

def wait_present(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present in the DOM.

    The element does not necessarily have to be visible.

    Useful when:
        - the element exists but is not yet displayed
        - another wait will handle visibility/clickability
    """
    return get_wait(driver, timeout).until(
        EC.presence_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """
    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """
    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 5. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """
    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 6. Wait until an element disappears from the DOM
# ------------------------------------------------------------

def wait_stale(driver, element, timeout=WAIT_SECONDS):
    """
    Wait until a previously obtained WebElement becomes stale.

    Useful when an existing element is replaced/removed after
    an operation such as saving a form.
    """
    return get_wait(driver, timeout).until(
        EC.staleness_of(element)
    )


# ------------------------------------------------------------
# 7. Wait for a locator to contain specific text
# ------------------------------------------------------------

def wait_text(driver, locator, text, timeout=WAIT_SECONDS):
    """
    Wait until an element contains the specified text.
    """
    return get_wait(driver, timeout).until(
        EC.text_to_be_present_in_element(
            locator,
            text,
        )
    )


# ------------------------------------------------------------
# 8. Wait for a specific attribute value
# ------------------------------------------------------------

def wait_attribute(
        driver,
        locator,
        attribute,
        value,
        timeout=WAIT_SECONDS,
):
    """
    Wait until an element's attribute contains the specified value.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator.

    attribute : str
        Name of the HTML attribute to inspect.

    value : str
        Value that must be present in the attribute.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    bool
        True when the attribute contains the specified value.
    """

    def attribute_contains(driver):
        try:
            element = driver.find_element(*locator)
            attribute_value = element.get_attribute(attribute)

            if attribute_value is None:
                return False

            return value in attribute_value

        except (
                NoSuchElementException,
                StaleElementReferenceException,
        ):
            return False

    return get_wait(driver, timeout).until(
        attribute_contains
    )


# ------------------------------------------------------------
# 9. Wait until an element is selected
# ------------------------------------------------------------

def wait_selected(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until a selectable element is selected.
    """
    return get_wait(driver, timeout).until(
        EC.element_located_to_be_selected(locator)
    )


# ------------------------------------------------------------
# 10. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
        driver,
        condition,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """
    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 11. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """
    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 12. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
        driver,
        popup_locator,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """
    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 13. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
        driver,
        locator,
        text,
        timeout=WAIT_SECONDS,
        clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    timeout : int | float
        Maximum number of seconds to wait.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """
    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element

In [ ]:
DELETE
ME!
# ============================================================
# TEST — Section 6: Reliable waits
# ============================================================
#
# This test verifies all reusable wait helpers defined in
# Section 6.
#
# The test uses a temporary local HTML page, so it does NOT:
#   - modify Eitaa
#   - add any contacts
#   - modify the MySQL database
#   - require an Eitaa login
#
# It uses the existing Selenium driver from Section 3.
#
# ============================================================

from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException

print("=" * 60)
print("SECTION 7 TEST — RELIABLE WAITS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verify required Section 7 objects exist
# ------------------------------------------------------------

required_functions = [
    "get_wait",
    "wait_present",
    "wait_visible",
    "wait_clickable",
    "wait_invisible",
    "wait_stale",
    "wait_text",
    "wait_attribute",
    "wait_selected",
    "wait_until",
    "wait_and_click",
    "wait_popup_closed",
    "wait_and_type",
]

missing_functions = [
    name
    for name in required_functions
    if name not in globals() or not callable(globals()[name])
]

if missing_functions:
    raise AssertionError(
        "Missing Section 7 functions: "
        + ", ".join(missing_functions)
    )

print("✓ All Section 7 functions are available")

# ------------------------------------------------------------
# 2. Verify Selenium driver exists
# ------------------------------------------------------------

if "driver" not in globals() or driver is None:
    raise RuntimeError(
        "Selenium driver was not found. "
        "Run Section 4 before running this test."
    )

print("✓ Selenium driver is available")

# ------------------------------------------------------------
# 3. Create a temporary test page
# ------------------------------------------------------------

test_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Section 7 Wait Test</title>
</head>

<body>

    <div id="present-element">
        Present element
    </div>

    <div id="visible-element">
        Visible element
    </div>

    <button id="click-button" type="button">
        Click me
    </button>

    <div id="text-element">
        Waiting for test text
    </div>

    <div id="attribute-element" class="test-element">
        Attribute test
    </div>

    <input
        id="text-input"
        type="text"
        value="old value"
    >

    <select id="select-element">
        <option value="one">One</option>
        <option value="two" selected>Two</option>
    </select>

    <div id="popup">
        Test popup
    </div>

    <div id="stale-element">
        This element will become stale
    </div>

    <div id="custom-element">
        Custom condition
    </div>

    <script>

        // Make the text condition available shortly after load.
        setTimeout(function () {
            document.getElementById("text-element").textContent =
                "Expected test text";
        }, 300);

        // Change the class attribute shortly after load.
        setTimeout(function () {
            document.getElementById("attribute-element")
                .className = "test-element active";
        }, 300);

        // Hide the popup shortly after load.
        setTimeout(function () {
            document.getElementById("popup").style.display = "none";
        }, 300);

        // Make the custom condition true shortly after load.
        setTimeout(function () {
            document.getElementById("custom-element")
                .setAttribute("data-ready", "true");
        }, 300);

    </script>

</body>
</html>
"""

# Load the HTML directly in Chrome.
driver.get(
    "data:text/html;charset=utf-8,"
    + test_html
)

# ------------------------------------------------------------
# 4. Define test locators
# ------------------------------------------------------------

PRESENT = (By.ID, "present-element")
VISIBLE = (By.ID, "visible-element")
CLICK_BUTTON = (By.ID, "click-button")
TEXT_ELEMENT = (By.ID, "text-element")
ATTRIBUTE_ELEMENT = (By.ID, "attribute-element")
TEXT_INPUT = (By.ID, "text-input")
SELECT_ELEMENT = (By.ID, "select-element")
POPUP = (By.ID, "popup")
STALE_ELEMENT = (By.ID, "stale-element")
CUSTOM_ELEMENT = (By.ID, "custom-element")

# ------------------------------------------------------------
# 5. Test get_wait()
# ------------------------------------------------------------

wait_object = get_wait(driver, timeout=5)

assert isinstance(
    wait_object,
    type(get_wait(driver, timeout=1))
)

print("✓ get_wait()")

# ------------------------------------------------------------
# 6. Test wait_present()
# ------------------------------------------------------------

element = wait_present(
    driver,
    PRESENT,
    timeout=5,
)

assert element is not None
assert element.get_attribute("id") == "present-element"

print("✓ wait_present()")

# ------------------------------------------------------------
# 7. Test wait_visible()
# ------------------------------------------------------------

element = wait_visible(
    driver,
    VISIBLE,
    timeout=5,
)

assert element.is_displayed()

print("✓ wait_visible()")

# ------------------------------------------------------------
# 8. Test wait_clickable()
# ------------------------------------------------------------

element = wait_clickable(
    driver,
    CLICK_BUTTON,
    timeout=5,
)

assert element.is_displayed()
assert element.is_enabled()

print("✓ wait_clickable()")

# ------------------------------------------------------------
# 9. Test wait_and_click()
# ------------------------------------------------------------

clicked_element = wait_and_click(
    driver,
    CLICK_BUTTON,
    timeout=5,
)

assert clicked_element.get_attribute("id") == "click-button"

print("✓ wait_and_click()")

# ------------------------------------------------------------
# 10. Test wait_text()
# ------------------------------------------------------------

wait_text(
    driver,
    TEXT_ELEMENT,
    "Expected test text",
    timeout=5,
)

actual_text = driver.find_element(
    *TEXT_ELEMENT
).text

assert "Expected test text" in actual_text

print("✓ wait_text()")

# ------------------------------------------------------------
# 11. Test wait_attribute()
# ------------------------------------------------------------

wait_attribute(
    driver,
    ATTRIBUTE_ELEMENT,
    "class",
    "active",
    timeout=5,
)

attribute_value = driver.find_element(
    *ATTRIBUTE_ELEMENT
).get_attribute("class")

assert "active" in attribute_value

print("✓ wait_attribute()")

# ------------------------------------------------------------
# 12. Test wait_selected()
# ------------------------------------------------------------

selected = wait_selected(
    driver,
    (By.CSS_SELECTOR, "#select-element option:checked"),
    timeout=5,
)

assert selected is not False

print("✓ wait_selected()")

# ------------------------------------------------------------
# 13. Test wait_and_type()
# ------------------------------------------------------------

typed_element = wait_and_type(
    driver,
    TEXT_INPUT,
    "Section 7 test",
    timeout=5,
)

assert typed_element.get_attribute("value") == "Section 7 test"

print("✓ wait_and_type()")

# ------------------------------------------------------------
# 14. Test wait_invisible()
# ------------------------------------------------------------

wait_invisible(
    driver,
    POPUP,
    timeout=5,
)

popup = driver.find_element(*POPUP)

assert not popup.is_displayed()

print("✓ wait_invisible()")

# ------------------------------------------------------------
# 15. Test wait_popup_closed()
# ------------------------------------------------------------

# The popup is already invisible, so this verifies that
# wait_popup_closed() correctly delegates to wait_invisible().

wait_popup_closed(
    driver,
    POPUP,
    timeout=5,
)

print("✓ wait_popup_closed()")

# ------------------------------------------------------------
# 16. Test wait_stale()
# ------------------------------------------------------------

# Create a fresh element specifically for this test.
driver.execute_script("""
    const element = document.createElement("div");
    element.id = "stale-test-element";
    element.textContent = "This element will become stale";
    document.body.appendChild(element);
""")

STALE_TEST_ELEMENT = (
    By.ID,
    "stale-test-element",
)

# Obtain the WebElement before it is removed.
stale_element = wait_present(
    driver,
    STALE_TEST_ELEMENT,
    timeout=5,
)

# Remove the element after a short delay.
driver.execute_script("""
    const element =
        document.getElementById("stale-test-element");

    setTimeout(function () {
        if (element) {
            element.remove();
        }
    }, 300);
""")

# Wait until the previously obtained WebElement becomes stale.
wait_stale(
    driver,
    stale_element,
    timeout=5,
)

print("✓ wait_stale()")

# ------------------------------------------------------------
# 17. Test wait_until()
# ------------------------------------------------------------

wait_until(
    driver,
    lambda d: d.find_element(
        *CUSTOM_ELEMENT
    ).get_attribute("data-ready") == "true",
    timeout=5,
)

assert driver.find_element(
    *CUSTOM_ELEMENT
).get_attribute("data-ready") == "true"

print("✓ wait_until()")

# ------------------------------------------------------------
# 18. Final result
# ------------------------------------------------------------

print()
print("=" * 60)
print("SECTION 7 TEST PASSED")
print("=" * 60)
print()
print("All Section 7 reliable-wait helpers passed successfully.")
print("The Selenium driver is still available for later sections.")

In [6]:
# ============================================================
# 7- Add Contact to Eitaa
# ============================================================

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)

# ------------------------------------------------------------
# Eitaa result
# ------------------------------------------------------------

UNREGISTERED_MESSAGE = (
    "The person with this phone number is not registered on Eitaa yet."
)

# ------------------------------------------------------------
# 7-1 Detect the unregistered-phone result
# ------------------------------------------------------------

def _get_unregistered_result(driver):
    """
    Return True when the Eitaa unregistered-phone toast is visible
    and contains the expected message.

    Return False when the toast is not currently present.
    """

    try:
        toast_elements = driver.find_elements(
            *Selectors.UNREGISTERED_TOAST
        )

        for toast in toast_elements:
            if not toast.is_displayed():
                continue

            text = toast.text.strip()

            if UNREGISTERED_MESSAGE in text:
                return True

    except WebDriverException:
        return False

    return False


# ------------------------------------------------------------
# 7-2 Detect result after Save
# ------------------------------------------------------------

def _wait_for_add_contact_result(driver):
    """
    Wait for exactly one result of the current Add Contact attempt.

    Possible return values:
        "skipped"  -> Eitaa reported that the phone is unregistered
        "success"  -> Add Contact window closed without that message

    If neither state can be determined within the wait period,
    wait_until() raises TimeoutException and the caller treats
    the attempt as failed.
    """

    def result_condition(d):

        # First priority: unregistered phone
        if _get_unregistered_result(d):
            return "skipped"

        # Registered/success case: the Add Contact window closes.
        try:
            close_buttons = d.find_elements(
                *Selectors.CLOSE_CONTACT_WINDOW
            )

            visible_close_buttons = [
                button
                for button in close_buttons
                if button.is_displayed()
            ]

            if not visible_close_buttons:
                return "success"

        except WebDriverException:
            return False

        return False

    return wait_until(
        driver,
        result_condition,
    )


# ------------------------------------------------------------
# 7-3 Close Add Contact window
# ------------------------------------------------------------


def _close_add_contact_window(driver):
    """
    Close the Add Contact window using the existing × button.

    First attempts to close the popup and waits for it to disappear.
    If it remains visible, performs one additional click.
    """

    # First close attempt
    wait_and_click(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    # Wait for the popup to close.
    try:
        wait_popup_closed(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5,
        )
        return

    except TimeoutException:
        pass

    # Retry the close action only if the popup remains visible.
    wait_and_click(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    wait_popup_closed(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )


# ------------------------------------------------------------
# 7-4 Best-effort cleanup after an operational failure
# ------------------------------------------------------------

def _close_add_contact_window_if_open(driver):
    """
    Best-effort cleanup after a failed attempt.

    Cleanup errors are intentionally ignored because the original
    Selenium/UI error must remain the result of this attempt.
    """

    try:
        elements = driver.find_elements(
            *Selectors.CLOSE_CONTACT_WINDOW
        )

        if not any(
                element.is_displayed()
                for element in elements
        ):
            return

        wait_and_click(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5,
        )

        try:
            wait_popup_closed(
                driver,
                Selectors.CLOSE_CONTACT_WINDOW,
                timeout=5,
            )
        except TimeoutException:
            pass

    except Exception:
        pass


# ------------------------------------------------------------
# 7-5 Enter phone number with Eitaa country prefix
# ------------------------------------------------------------

from selenium.webdriver.common.keys import Keys


def _enter_phone_number(driver, phone_normalized):
    """
    Clear the Eitaa phone field, verify that the previous phone
    number/country prefix has been removed, then enter:

        98 + normalized phone number

    The '+' sign is NOT entered manually because Eitaa adds it
    automatically.
    """

    # --------------------------------------------------------
    # 1. Wait for the phone field
    # --------------------------------------------------------

    phone_field = wait_visible(
        driver,
        Selectors.PHONE,
    )

    # --------------------------------------------------------
    # 2. Clear the existing contents
    #
    # Use both Selenium's clear() and keyboard selection.
    # This is more reliable for phone inputs that use client-side
    # formatting/masking.
    # --------------------------------------------------------

    phone_field.click()

    phone_field.clear()

    phone_field.send_keys(
        Keys.CONTROL,
        "a",
    )

    phone_field.send_keys(
        Keys.BACKSPACE,
    )

    # --------------------------------------------------------
    # 3. Verify that the old 98 prefix is no longer present
    #
    # We intentionally do NOT require value == "" because Eitaa's
    # phone field may maintain its own country-code state.
    # The important condition is that the old phone digits and
    # country prefix are no longer present in the editable value.
    # --------------------------------------------------------

    def phone_field_is_cleared(d):
        try:
            element = d.find_element(
                *Selectors.PHONE
            )

            value = element.get_attribute("value")

            if value is None:
                return True

            value = value.strip()

            # The field must not contain the old country prefix
            # as an entered phone-number value.
            #
            # Eitaa may keep its own "+" / country-code UI state,
            # so an absolutely empty value is not required here.

            return (
                    value == ""
                    or value == "+"
            )

        except WebDriverException:
            return False

    wait_until(
        driver,
        phone_field_is_cleared,
        timeout=5,
    )

    # --------------------------------------------------------
    # 4. Prepare normalized phone number
    # --------------------------------------------------------

    phone_number = str(phone_normalized).strip()

    if phone_number.startswith("98"):
        raise ValueError(
            "phone_normalized must not contain the country code 98."
        )

    # --------------------------------------------------------
    # 5. Enter 98 + normalized phone number
    #
    # Do NOT enter '+' manually.
    # Eitaa adds '+' automatically.
    # --------------------------------------------------------

    phone_field = wait_visible(
        driver,
        Selectors.PHONE,
    )

    phone_field.send_keys(
        "98" + phone_number
    )


# ============================================================
# 7-6 Main Section 8 function
# ============================================================

def add_contact(
        full_name,
        phone_normalized,
):
    """
    Process exactly ONE contact in exactly ONE attempt.

    Parameters
    ----------

    full_name : str
        Contact name.

    phone_normalized : str
        Normalized Iranian phone number supplied by the DataFrame.

    Returns
    -------
    str
        One of:

            "success"
            "skipped"
            "failed"

    """

    try:
        # ----------------------------------------------------
        # 1. Open Contacts area if necessary
        # ----------------------------------------------------


        # ----------------------------------------------------
        # 2. Open Add Contact window
        # ----------------------------------------------------

        wait_and_click(
            driver,
            Selectors.ADD_CONTACT_BUTTON,
        )

        # ----------------------------------------------------
        # 3. Enter contact name
        # ----------------------------------------------------

        wait_and_type(
            driver,
            Selectors.FIRST_NAME,
            str(full_name),
        )

        # ----------------------------------------------------
        # 4. Enter normalized phone number
        #
        # IMPORTANT:
        # Do NOT use wait_and_type() here because it clears the
        # field and immediately types the normalized number.
        #
        # Instead:
        #   1. clear the field
        #   2. verify it is empty
        #   3. enter 98
        #   4. enter the normalized number
        #
        # The '+' sign is added automatically by Eitaa.
        # ----------------------------------------------------

        _enter_phone_number(
            driver,
            phone_normalized,
        )

        # ----------------------------------------------------
        # 5. Submit Add Contact form
        # ----------------------------------------------------

        wait_and_click(
            driver,
            Selectors.SAVE_BUTTON,
        )

        # ----------------------------------------------------
        # 6. Determine the result of THIS attempt
        # ----------------------------------------------------

        result = _wait_for_add_contact_result(driver)

        # ----------------------------------------------------
        # 7. Registered contact
        # ----------------------------------------------------

        if result == "success":
            return "success"

        # ----------------------------------------------------
        # 8. Unregistered phone number
        # ----------------------------------------------------

        if result == "skipped":
            # The unregistered toast has already been reliably
            # detected. Close the Add Contact operation using
            # the existing close-button mechanism.
            _close_add_contact_window(driver)

            return "skipped"

        # ----------------------------------------------------
        # 9.Defensive guard
        # ----------------------------------------------------

        raise RuntimeError(
            f"Unexpected Section 8 result: {result!r}"
        )

    except Exception as exc:
        # Best-effort UI cleanup
        _close_add_contact_window_if_open(driver)

        # Report the error without accessing the database
        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        print(
            f"Section 7 failed for full_name={full_name}: "
            f"{error_message}"
        )

        return "failed"

In [7]:
# ============================================================
# Section 8: Main Processing Loop
# ============================================================

success_count = 0
skipped_count = 0
failed_count = 0

print("=" * 60)
print("Section 8: Main Processing Loop")
print("=" * 60)

for _, row in df.iterrows():
    contact_id = row["contact_id"]
    full_name = row["full_name"]
    phone_normalized = row["phone"]

    try:
        result = add_contact(
            full_name=full_name,
            phone_normalized=phone_normalized
        )

    except Exception as exc:
        failed_count += 1

        print(
            f"Contact ID {contact_id} failed: "
            f"{type(exc).__name__}: {exc}"
        )
        continue

    if result == "success":
        success_count += 1

    elif result == "skipped":
        skipped_count += 1

    elif result == "failed":
        failed_count += 1

    else:
        # Handle an unexpected return value.
        failed_count += 1

        print(
            f"Contact ID {contact_id} failed: "
            f"Unexpected return value {result!r}"
        )

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("Section 8: Processing Summary")
print("=" * 60)

print(f"Total contacts: {len(df)}")
print(f"Success:        {success_count}")
print(f"Skipped:        {skipped_count}")
print(f"Failed:         {failed_count}")

print("=" * 60)
print("Section 8 completed.")

Section 8: Main Processing Loop

Section 8: Processing Summary
Total contacts: 44
Success:        12
Skipped:        32
Failed:         0
Section 8 completed.
